# 05j-n — Refit del decoder sul supporto rigenerativo

Manteniamo H2 congelato e la famiglia direct-tree già registrata. Normalizzazione, PCA e pesi del decoder sono rifittati soltanto sul fit interno formato da supporto 05j-h e 05j-m. La calibrazione interna sceglie i checkpoint; 05j-i è development post-freeze. Il fresh test 05j-m resta sigillato.

## 1. Checkout e runtime

In [ ]:
import os,subprocess,sys
from pathlib import Path
WORKSPACE=Path('/kaggle/working/hayflow_workspace'); ELM_REPO=WORKSPACE/'elmneuron'
if not ELM_REPO.exists(): subprocess.run(['git','clone','https://github.com/Zagred47/giada.git',str(ELM_REPO)],check=True)
subprocess.run(['git','-C',str(ELM_REPO),'fetch','origin','main'],check=True); subprocess.run(['git','-C',str(ELM_REPO),'checkout','--detach','FETCH_HEAD'],check=True); subprocess.run([sys.executable,'-m','pip','install','-q','h5py','pandas','pyarrow','pyyaml'],check=True)
sys.path.insert(0,str(ELM_REPO)); REVISION=subprocess.check_output(['git','-C',str(ELM_REPO),'rev-parse','HEAD'],text=True).strip(); print('Revision:',REVISION)

In [ ]:
import h5py,json,numpy as np,pandas as pd,pyarrow,torch,yaml
assert torch.cuda.is_available(),'Attiva una GPU Kaggle prima di eseguire 05j-n.'
print({'torch':torch.__version__,'cuda':torch.cuda.get_device_name(0)})

## 2. Catena esatta degli artefatti

In [ ]:
import hashlib,shutil,zipfile
from src.hayflow_model.hines_state_normalization_repair import EXPECTED_05H_INDEX_SHA256
from src.hayflow_model.hines_netcon_semantic_repair import EXPECTED_05I_INDEX_SHA256
from src.hayflow_model.hines_synaptic_domain_repair import EXPECTED_05IB_INDEX_SHA256
from src.hayflow_model.hines_repaired_representation_recheck import EXPECTED_05IC_INDEX_SHA256
from src.hayflow_model.hines_repaired_representation_revision import EXPECTED_05J_INDEX_SHA256
from src.hayflow_model.hines_spatial_support_revision import EXPECTED_05JB_INDEX_SHA256
from src.hayflow_model.hines_trainable_topology_canary import EXPECTED_05JC_INDEX_SHA256
from src.hayflow_model.hines_architecture_reassessment import EXPECTED_05JD_INDEX_SHA256
from src.hayflow_model.hines_region_mechanism_experts import EXPECTED_05JE_INDEX_SHA256
from src.hayflow_model.hines_regenerative_state_decomposition import EXPECTED_05JF_INDEX_SHA256
from src.hayflow_model.hines_regenerative_support_expansion import EXPECTED_05JG_INDEX_SHA256
from src.hayflow_model.hines_regenerative_confirmation import EXPECTED_05JH_INDEX_SHA256,EXPECTED_05JI_INDEX_SHA256
from src.hayflow_model.hines_voltage_objective_reassessment import EXPECTED_05JJ_INDEX_SHA256
from src.hayflow_model.hines_residual_safety_gate import EXPECTED_05JK_INDEX_SHA256
from src.hayflow_model.hines_regenerative_decoder_refit import EXPECTED_05JL_INDEX_SHA256,EXPECTED_05JM_INDEX_SHA256
INPUT_ROOT=Path('/kaggle/input')
def extract_zip_safely(source,destination):
 source,destination=Path(source),Path(destination); marker=destination/'.source_size'; stamp=str(source.stat().st_size)
 if marker.is_file() and marker.read_text().strip()==stamp:return destination
 if destination.exists():shutil.rmtree(destination)
 destination.mkdir(parents=True);root=destination.resolve()
 with zipfile.ZipFile(source) as archive:
  for member in archive.infolist():
   target=(destination/member.filename).resolve();assert target==root or root in target.parents,member.filename
  archive.extractall(destination)
 marker.write_text(stamp);return destination
def index_matches(path,expected):
 path=Path(path)
 try:
  if path.is_file():
   with zipfile.ZipFile(path) as archive:
    names=[n for n in archive.namelist() if n.replace('\\','/').endswith('artifact_index.json')];return len(names)==1 and hashlib.sha256(archive.read(names[0])).hexdigest()==expected
  return any(hashlib.sha256(p.read_bytes()).hexdigest()==expected for p in path.rglob('artifact_index.json'))
 except (OSError,zipfile.BadZipFile):return False
def artifact(env,name,marker,expected):
 candidates=([Path(os.environ[env]).expanduser()] if os.environ.get(env) else [])+list(INPUT_ROOT.rglob(name))+[p.parent for p in INPUT_ROOT.rglob(marker)];valid=[p.resolve() for p in candidates if p.exists() and index_matches(p,expected)];assert valid,f'{name} non trovato o incompatibile: {[str(p) for p in candidates]}';return valid[0]
def marker_artifact(name,marker):
 candidates=list(INPUT_ROOT.rglob(name))+[p.parent for p in INPUT_ROOT.rglob(marker)];found=next((p.resolve() for p in candidates if p.exists()),None);assert found is not None,f'{name} non trovato.';return found
topup_candidates=([Path(os.environ['HAYFLOW_TOPUP_V3']).expanduser()] if os.environ.get('HAYFLOW_TOPUP_V3') else [])+list(INPUT_ROOT.rglob('hayflow_bap_validation_support_topup_v3.zip'))+[p.parent for p in INPUT_ROOT.rglob('composite_dataset_manifest.json')];TOPUP_SOURCE=next((p.resolve() for p in topup_candidates if p.exists()),None);assert TOPUP_SOURCE is not None,'Top-up BAP v3 non trovato.';TOPUP_ROOT=extract_zip_safely(TOPUP_SOURCE,'/kaggle/working/hayflow05jn_topup') if TOPUP_SOURCE.is_file() else TOPUP_SOURCE;manifests=list(Path(TOPUP_ROOT).rglob('composite_dataset_manifest.json'));assert len(manifests)==1,manifests;COMPOSITE_MANIFEST=manifests[0]
base_candidates=([Path(os.environ['HAYFLOW_BASE_DATASET']).expanduser()] if os.environ.get('HAYFLOW_BASE_DATASET') else [])+[p.parent for p in INPUT_ROOT.rglob('transition_dataset.h5') if 'targeted' in str(p).lower() and 'topup' not in str(p).lower() and 'confirmation' not in str(p).lower() and 'training-support' not in str(p).lower()]+[p for p in INPUT_ROOT.rglob('archive.zip') if 'hayflow-targeted-transition-dataset' in str(p).lower()];BASE_SOURCE=next((p.resolve() for p in base_candidates if p.exists()),None);assert BASE_SOURCE is not None,'Dataset base non trovato.'
CHECKPOINT_05B_SOURCE=marker_artifact('hayflow_hines_canary_v2.zip','canary_models.pt');CHECKPOINT_05B_SOURCE=CHECKPOINT_05B_SOURCE.parent if CHECKPOINT_05B_SOURCE.name=='checkpoints' else CHECKPOINT_05B_SOURCE
ARTIFACT_05C_SOURCE=marker_artifact('hayflow_hines_causal_isolation.zip','checkpoint_forensics.json');ARTIFACT_05D_SOURCE=marker_artifact('hayflow_hines_residual_conditioning.zip','free_residual_report.json');ARTIFACT_05E_SOURCE=marker_artifact('hayflow_hines_segment_capacity.zip','capacity_probe_report.json');ARTIFACT_05F_SOURCE=marker_artifact('hayflow_hines_segment_micro_canary.zip','micro_canary_report.json');ARTIFACT_05G_SOURCE=marker_artifact('hayflow_hines_optimization_audit.zip','optimization_support.json')
specs=[('05H','hayflow_hines_representation_forensics.zip','representation_forensics_config.json',EXPECTED_05H_INDEX_SHA256),('05I','hayflow_hines_state_normalization_repair.zip','state_normalization_repair_config.json',EXPECTED_05I_INDEX_SHA256),('05IB','hayflow_hines_netcon_semantic_state_repair.zip','netcon_semantic_repair_config.json',EXPECTED_05IB_INDEX_SHA256),('05IC','hayflow_hines_synaptic_domain_repair.zip','synaptic_domain_repair_config.json',EXPECTED_05IC_INDEX_SHA256),('05J','hayflow_hines_repaired_representation_recheck.zip','repaired_representation_recheck_config.json',EXPECTED_05J_INDEX_SHA256),('05JB','hayflow_hines_repaired_representation_revision.zip','repaired_representation_revision_config.json',EXPECTED_05JB_INDEX_SHA256),('05JC','hayflow_hines_spatial_support_revision.zip','spatial_support_revision_config.json',EXPECTED_05JC_INDEX_SHA256),('05JD','hayflow_hines_trainable_topology_decoder_micro_canary.zip','trainable_topology_canary_config.json',EXPECTED_05JD_INDEX_SHA256),('05JE','hayflow_hines_architecture_reassessment.zip','architecture_reassessment_config.json',EXPECTED_05JE_INDEX_SHA256),('05JF','hayflow_hines_region_mechanism_expert_revision.zip','region_mechanism_expert_config.json',EXPECTED_05JF_INDEX_SHA256),('05JG','hayflow_hines_regenerative_state_decomposition.zip','state_target_decomposition_config.json',EXPECTED_05JG_INDEX_SHA256),('05JH','hayflow_hines_regenerative_support_expansion.zip','regenerative_support_expansion_config.json',EXPECTED_05JH_INDEX_SHA256),('05JI','hayflow_regenerative_confirmation_support.zip','confirmation_plan.json',EXPECTED_05JI_INDEX_SHA256),('05JJ','hayflow_hines_regenerative_confirmation.zip','independent_confirmation_config.json',EXPECTED_05JJ_INDEX_SHA256),('05JK','hayflow_hines_voltage_objective_reassessment.zip','voltage_objective_reassessment_config.json',EXPECTED_05JK_INDEX_SHA256),('05JL','hayflow_hines_residual_safety_gate.zip','residual_safety_gate_config.json',EXPECTED_05JL_INDEX_SHA256),('05JM','hayflow_regenerative_training_support.zip','acquisition_contract.json',EXPECTED_05JM_INDEX_SHA256)]
for key,name,marker,expected in specs:globals()[f'ARTIFACT_{key}_SOURCE']=artifact(f'HAYFLOW_{key}_ARTIFACT',name,marker,expected)
print({'base':str(BASE_SOURCE),'05j-l':str(ARTIFACT_05JL_SOURCE),'05j-m':str(ARTIFACT_05JM_SOURCE)})

## 3. Preflight del dataset composito

In [ ]:
import time
from src.hayflow_data import prepare_composite_flowmap_bundle
started,last={},{}
def progress(name,done,total):
 now=time.monotonic();started.setdefault(name,now);pct=int(100*done/total)
 if pct>=last.get(name,-5)+5 or done==total:
  rate=done/max(now-started[name],1e-9);eta=(total-done)/max(rate,1e-9);print(f'[HayFlow 05j-n][SHA-256 {name}] {pct}% ETA {eta/60:.1f} min',flush=True);last[name]=pct
bundle=prepare_composite_flowmap_bundle(COMPOSITE_MANIFEST,base_source=BASE_SOURCE,progress=progress);display({'valid':bundle.manifest['valid'],'fingerprint':bundle.fingerprint,'transitions':bundle.transition_count});assert bundle.manifest['valid'] and bundle.transition_count==29880

## 4. Sessione e contratto 05j-n

In [ ]:
from src.hayflow_model import *
def config(name):return yaml.safe_load((ELM_REPO/'configs/hayflow'/name).read_text())
b=config('hayflow_hines_optimization_audit.yml');f=config('hayflow_hines_representation_forensics.yml');r=config('hayflow_hines_state_normalization_repair.yml');n=config('hayflow_hines_netcon_semantic_repair.yml');d=config('hayflow_hines_synaptic_domain_repair.yml');rc=config('hayflow_hines_repaired_representation_recheck.yml');rv=config('hayflow_hines_repaired_representation_revision.yml');sp=config('hayflow_hines_spatial_support_revision.yml');tp=config('hayflow_hines_trainable_topology_canary.yml');ra=config('hayflow_hines_architecture_reassessment.yml');ex=config('hayflow_hines_region_mechanism_experts.yml');dc=config('hayflow_hines_regenerative_state_decomposition.yml');se=config('hayflow_hines_regenerative_support_expansion.yml');cf=config('hayflow_hines_regenerative_confirmation.yml');oa=config('hayflow_hines_voltage_objective_reassessment.yml');sg=config('hayflow_hines_residual_safety_gate.yml');rf=config('hayflow_hines_regenerative_decoder_refit.yml')
model_config=HinesPrototypeExperimentConfig.from_mapping(b['model_experiment']);isolation_config=HinesIsolationConfig.from_mapping(b['isolation']);conditioning_config=HinesConditioningConfig.from_mapping(b['conditioning']);capacity_config=HinesCapacityConfig.from_mapping(b['capacity']);canary_config=HinesSegmentCanaryConfig.from_mapping(b['micro_canary']);audit_config=HinesOptimizationAuditConfig.from_mapping(b['optimization_audit']);representation_config=HinesRepresentationForensicsConfig.from_mapping(f['representation_forensics']);repair_config=HinesStateNormalizationRepairConfig.from_mapping(r['state_normalization_repair']);netcon_config=HinesNetConSemanticRepairConfig.from_mapping(n['netcon_semantic_repair']);domain_config=HinesSynapticDomainRepairConfig.from_mapping(d['synaptic_domain_repair']);recheck_config=HinesRepairedRepresentationRecheckConfig.from_mapping(rc['repaired_representation_recheck']);revision_config=HinesRepairedRepresentationRevisionConfig.from_mapping(rv['repaired_representation_revision']);spatial_config=HinesSpatialSupportRevisionConfig.from_mapping(sp['spatial_support_revision']);topology_config=HinesTrainableTopologyCanaryConfig.from_mapping(tp['trainable_topology_canary']);reassessment_config=HinesArchitectureReassessmentConfig.from_mapping(ra['architecture_reassessment']);expert_config=HinesRegionMechanismExpertConfig.from_mapping(ex['region_mechanism_experts']);decomposition_config=HinesRegenerativeStateDecompositionConfig.from_mapping(dc['regenerative_state_decomposition']);support_expansion_config=HinesRegenerativeSupportExpansionConfig.from_mapping(se['regenerative_support_expansion']);confirmation_config=HinesRegenerativeConfirmationConfig.from_mapping(cf['independent_confirmation']);objective_config=HinesVoltageObjectiveReassessmentConfig.from_mapping(oa['voltage_objective_reassessment']);safety_config=HinesResidualSafetyGateConfig.from_mapping(sg['residual_safety_gate']);refit_config=HinesRegenerativeDecoderRefitConfig.from_mapping(rf['regenerative_decoder_refit'])
OUTPUT_DIR=Path('/kaggle/working/artifacts/hayflow_hines_regenerative_decoder_refit');shutil.rmtree(OUTPUT_DIR,ignore_errors=True)
session=HinesRegenerativeDecoderRefit(bundle,OUTPUT_DIR,model_config,isolation_config,conditioning_config,capacity_config,canary_config,audit_config,representation_config,CHECKPOINT_05B_SOURCE,ARTIFACT_05C_SOURCE,ARTIFACT_05D_SOURCE,ARTIFACT_05E_SOURCE,ARTIFACT_05F_SOURCE,ARTIFACT_05G_SOURCE,repair_config=repair_config,artifact_05h_source=ARTIFACT_05H_SOURCE,netcon_config=netcon_config,artifact_05i_source=ARTIFACT_05I_SOURCE,domain_config=domain_config,artifact_05ib_source=ARTIFACT_05IB_SOURCE,recheck_config=recheck_config,artifact_05ic_source=ARTIFACT_05IC_SOURCE,revision_config=revision_config,artifact_05j_source=ARTIFACT_05J_SOURCE,spatial_config=spatial_config,artifact_05jb_source=ARTIFACT_05JB_SOURCE,topology_config=topology_config,artifact_05jc_source=ARTIFACT_05JC_SOURCE,reassessment_config=reassessment_config,artifact_05jd_source=ARTIFACT_05JD_SOURCE,expert_config=expert_config,artifact_05je_source=ARTIFACT_05JE_SOURCE,decomposition_config=decomposition_config,artifact_05jf_source=ARTIFACT_05JF_SOURCE,support_expansion_config=support_expansion_config,artifact_05jg_source=ARTIFACT_05JG_SOURCE,confirmation_config=confirmation_config,artifact_05jh_source=ARTIFACT_05JH_SOURCE,artifact_05ji_source=ARTIFACT_05JI_SOURCE,objective_config=objective_config,artifact_05jj_source=ARTIFACT_05JJ_SOURCE,safety_config=safety_config,artifact_05jk_source=ARTIFACT_05JK_SOURCE,refit_config=refit_config,artifact_05jl_source=ARTIFACT_05JL_SOURCE,artifact_05jm_source=ARTIFACT_05JM_SOURCE,code_revision=REVISION)
prepare_report=session.prepare_regenerative_decoder_refit();display({'revision':REVISION,'05j-m':prepare_report['artifact_05jm'],'training_store':prepare_report['training_store']});assert prepare_report['training_store']['valid'];assert not prepare_report['fresh_test_inputs_extracted'] and not prepare_report['fresh_test_outcomes_generated']

## 5. Ricostruzione esatta dei componenti congelati

In [ ]:
session.apply_verified_synaptic_domain_normalizer();session.build_expanded_train_support();session.prepare_expanded_spatial_features();design=session.prepare_topology_canary_designs();session.fit_fixed_tree_ridge_baseline();reconstruction=session.reconstruct_frozen_checkpoints(metric_atol=expert_config.checkpoint_reconstruction_metric_atol);session.build_regenerative_support();session.prepare_expanded_regenerative_roles();expanded=session.reconstruct_expanded_direct_tree_ensemble();external=session.prepare_external_confirmation_roles();display({'design':design['valid'],'reconstruction':reconstruction['valid'],'expanded':expanded['valid'],'development_pairs':external['pair_count']});assert design['valid'] and reconstruction['valid'] and expanded['valid'] and external['valid']

## 6. Split interno outcome-blind e rifit del decoder

In [ ]:
role_report=session.prepare_refit_roles();display(role_report);assert role_report['valid'];assert role_report['combined_internal_fit_pair_count']==120 and role_report['combined_internal_calibration_pair_count']==30 and role_report['development_pair_count']==24;assert not role_report['development_used_to_fit_representation'] and not role_report['fresh_test_loaded']
refit_report=session.run_decoder_refit();display(pd.DataFrame([{'seed':r['seed'],'best_epoch':r['best_epoch'],'internal_gain':r['internal_improvement_vs_h2_fraction'],'development_gain':r['development_improvement_vs_best_baseline_fraction'],'development_rmse_mv':r['roles']['development']['aggregate_voltage_rmse_mv'],'passed':r['run_passed']} for r in refit_report['runs']]));display({'passing_seeds':refit_report['passing_seed_count'],'robust_gate_passed':refit_report['robust_gate_passed']});assert not refit_report['development_used_for_checkpoint_selection'];assert not refit_report['fresh_test_inputs_extracted'] and not refit_report['fresh_test_outcomes_generated']

## 7. Decisione metodologica

In [ ]:
final_report=session.finalize_decoder_refit(role_report,refit_report);display({'valid':final_report['valid'],'diagnosis':final_report['diagnosis'],'fresh_test_generation_authorized':final_report['fresh_test_generation_authorized'],'next_step':final_report['next_step']});assert final_report['valid'];assert not final_report['candidate_model_authorized'] and not final_report['micro_rollout_authorized'] and not final_report['full_training_authorized'];assert not final_report['methodology']['fresh_test_inputs_extracted'] and not final_report['methodology']['fresh_test_outcomes_generated']

## 8. Crea e scarica lo ZIP

In [ ]:
from shutil import make_archive
import base64
from IPython.display import Javascript,display
zip_path=Path(make_archive('/kaggle/working/hayflow_hines_regenerative_decoder_refit','zip',root_dir=OUTPUT_DIR.parent,base_dir=OUTPUT_DIR.name));payload=base64.b64encode(zip_path.read_bytes()).decode('ascii');filename=zip_path.name
display(Javascript(f"""const binary=atob('{payload}');const bytes=new Uint8Array(binary.length);for(let i=0;i<binary.length;i++)bytes[i]=binary.charCodeAt(i);const blob=new Blob([bytes],{{type:'application/zip'}});const url=URL.createObjectURL(blob);const a=document.createElement('a');a.href=url;a.download='{filename}';document.body.appendChild(a);a.click();a.remove();setTimeout(()=>URL.revokeObjectURL(url),60000);"""));print({'zip':str(zip_path),'size_mib':round(zip_path.stat().st_size/2**20,2),'download':'avviato dal browser'})